In [2]:
from model_ranking import (
    loader_classes,
)
from pytorch3dunet.datasets.utils import get_test_loaders

INFO: P [MainThread] 2025-10-29 15:48:51,065 plantseg - Logger configured at initialisation. PlantSeg logger name: plantseg


/g/kreshuk/talks/pytorch-3dunet/pytorch3dunet/unet3d/utils.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/g/kreshuk/talks/miniforge3/envs/model-rank-local2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
loader_config = {
    'dataset': 'StandardHDF5Dataset',
    'batch_size': 5,
    'num_workers': 8,
    'raw_internal_path': 'volumes/raw',
    'label_internal_path': 'volumes/labels/expanded_cells_with_ignore',
    'global_normalization': True,
    'global_percentiles': [5.0, 95.0],
    'output_dir': '/g/kreshuk/talks/consistency_results/Instance_segmentation/Cells/FlyWing_to_FlyWing_gap/consistency/P_fullslice/fw_model_Unetr3/norm_50_950/none/predictions',
    'test': {
        'file_paths': [
            '/scratch/talks/data/FlyWing/GT/test/per03.h5'
        ],
        'slice_builder': {
            'name': 'SingleZSliceBuilder',
            'patch_shape': [1, 640, 640],
            'stride_shape': [1, 640, 640],
            'halo_shape': [0, 0, 0]
        },
        'transformer': {
            'raw': [
                {'name': 'PercentileNormalizer'},
                {'name': 'ToTensor', 'expand_dims': True}
            ]
        },
        'roi': None
    }
}

In [5]:
dataset_class = loader_classes(loader_config['dataset'])
pred_datasets = dataset_class.create_datasets(loader_config, 'test')
dataset = pred_datasets[0]
print(len(dataset))

2025-10-29 15:49:07,643 [MainThread] INFO HDF5Dataset - Loading test set from: /scratch/talks/data/FlyWing/GT/test/per03.h5...
2025-10-29 15:49:07,647 [MainThread] INFO HDF5Dataset - Auto-calculated padding: [0, 1, 0] for patch_shape: [1, 640, 640] and volume_shape: (160, 639, 765)
2025-10-29 15:49:07,648 [MainThread] INFO HDF5Dataset - Calculating mean and std of the raw data...
2025-10-29 15:49:07,647 [MainThread] INFO HDF5Dataset - Auto-calculated padding: [0, 1, 0] for patch_shape: [1, 640, 640] and volume_shape: (160, 639, 765)
2025-10-29 15:49:07,648 [MainThread] INFO HDF5Dataset - Calculating mean and std of the raw data...
Global mean: 69.43447588785904, global std: 48.09484884511062
2025-10-29 15:49:09,898 [MainThread] INFO Dataset - Slice builder config: {'name': 'SingleZSliceBuilder', 'patch_shape': [1, 640, 640], 'stride_shape': [1, 640, 640], 'halo_shape': [0, 0, 0]}
2025-10-29 15:49:09,899 [MainThread] INFO HDF5Dataset - Number of patches: 160
160
Global mean: 69.43447588

In [6]:
raw, _ = dataset[0]
print(raw.shape)

torch.Size([1, 1, 640, 640])


In [1]:
# Check the actual shape after padding
print(f"Dataset auto_padding: {dataset.auto_padding}")
print(f"Dataset needs_padding: {dataset._needs_padding}")
# Load the raw data to check actual shape
import h5py
with h5py.File('/scratch/talks/data/FlyWing/GT/test/per03.h5', 'r') as f:
    raw_shape = f['volumes/raw'].shape
    print(f"Original raw shape: {raw_shape}")
    
# Check the cached raw data shape
if dataset._raw is not None:
    print(f"Cached raw shape after padding: {dataset._raw.shape}")
else:
    # Force loading
    _ = dataset.get_raw_patch(slice(None))
    print(f"Cached raw shape after padding: {dataset._raw.shape}")

NameError: name 'dataset' is not defined

In [16]:
# Reload the module and recreate the dataset to test the fix
import importlib
import sys

# Find and reload the hdf5 module
for module_name in list(sys.modules.keys()):
    if 'hdf5' in module_name or 'pytorch3dunet' in module_name:
        if module_name in sys.modules:
            del sys.modules[module_name]

# Reimport
from model_ranking import loader_classes

# Recreate dataset
dataset_class = loader_classes(loader_config['dataset'])
pred_datasets = dataset_class.create_datasets(loader_config, 'test')
dataset_new = pred_datasets[0]
print(f"Number of patches: {len(dataset_new)}")

# Check the padding
print(f"New auto_padding: {dataset_new.auto_padding}")
if dataset_new._raw is not None or True:
    _ = dataset_new.get_raw_patch(slice(None))
    print(f"New cached raw shape after padding: {dataset_new._raw.shape}")

# Test getting a patch
raw_new, _ = dataset_new[0]
print(f"New raw patch shape: {raw_new.shape}")

2025-10-29 15:31:32,841 [MainThread] INFO HDF5Dataset - Loading test set from: /scratch/talks/data/FlyWing/GT/test/per03.h5...
2025-10-29 15:31:32,841 [MainThread] INFO HDF5Dataset - Loading test set from: /scratch/talks/data/FlyWing/GT/test/per03.h5...
2025-10-29 15:31:32,844 [MainThread] INFO HDF5Dataset - Auto-calculated padding: [0, 0, 0] for patch_shape: [1, 640, 640] and volume_shape: (160, 639, 765)
2025-10-29 15:31:32,844 [MainThread] INFO HDF5Dataset - Auto-calculated padding: [0, 0, 0] for patch_shape: [1, 640, 640] and volume_shape: (160, 639, 765)
2025-10-29 15:31:32,846 [MainThread] INFO HDF5Dataset - Calculating mean and std of the raw data...
2025-10-29 15:31:32,846 [MainThread] INFO HDF5Dataset - Calculating mean and std of the raw data...
Global mean: 69.46046953471007, global std: 48.11441160165768
2025-10-29 15:31:35,086 [MainThread] INFO Dataset - Slice builder config: {'name': 'SingleZSliceBuilder', 'patch_shape': [1, 640, 640], 'stride_shape': [1, 640, 640], 'halo

IndexError: list index out of range

In [17]:
# Let's create a wrapper directly to see what shape it reports
from pytorch3dunet.datasets.hdf5 import FastShapeWrapper
wrapper = FastShapeWrapper(
    '/scratch/talks/data/FlyWing/GT/test/per03.h5',
    'volumes/raw',
    None,
    [0, 1, 0]  # The auto_padding that should be calculated
)
print(f"Wrapper shape: {wrapper.shape}")
print(f"Wrapper ndim: {wrapper.ndim}")

Wrapper shape: (160, 641, 765)
Wrapper ndim: 3


In [18]:
# Now let's check what the dataset's auto_padding actually is when it's created
print(f"Old dataset auto_padding: {dataset.auto_padding}")
print(f"Old dataset needs_padding: {dataset._needs_padding}")

# And check what the raw_slices look like
print(f"First few raw slices: {dataset.raw_slices[:3]}")

Old dataset auto_padding: [0, 1, 0]
Old dataset needs_padding: True
First few raw slices: [(slice(0, 1, None), slice(0, 640, None), slice(62, 702, None)), (slice(1, 2, None), slice(0, 640, None), slice(62, 702, None)), (slice(2, 3, None), slice(0, 640, None), slice(62, 702, None))]


In [19]:
# Let's check the actual cached raw data shape and what gets extracted
print(f"Cached raw shape (should be padded): {dataset._raw.shape if dataset._raw is not None else 'Not loaded yet'}")

# Get the raw slice indices for the first patch
raw_idx = dataset.raw_slices[0]
print(f"Raw slice for patch 0: {raw_idx}")

# Try to extract using this slice
if dataset._raw is not None:
    extracted = dataset._raw[raw_idx]
    print(f"Extracted shape: {extracted.shape}")

Cached raw shape (should be padded): (160, 641, 765)
Raw slice for patch 0: (slice(0, 1, None), slice(0, 640, None), slice(62, 702, None))
Extracted shape: (1, 640, 640)


In [20]:
# Let's manually go through the getitem process
idx = 0
raw_idx = dataset.raw_slices[idx]
print(f"Getting patch at index {idx}")
print(f"Raw slice: {raw_idx}")

# Check if this is 4D (has channel dimension)
print(f"Length of raw_idx: {len(raw_idx)}")
print(f"Phase: {dataset.phase}")

# In test phase, there's special handling for 4D slices
if dataset.phase == 'test' and len(raw_idx) == 4:
    print("4D case - discarding channel dimension")
    raw_idx = raw_idx[1:]
    print(f"Adjusted raw_idx: {raw_idx}")

# Now get the raw patch
raw_patch = dataset.get_raw_patch(raw_idx)
print(f"Raw patch shape before transform: {raw_patch.shape}")

# Apply transform
raw_patch_transformed = dataset.raw_transform(raw_patch)
print(f"Raw patch shape after transform: {raw_patch_transformed.shape}")

Getting patch at index 0
Raw slice: (slice(0, 1, None), slice(0, 640, None), slice(62, 702, None))
Length of raw_idx: 3
Phase: test
Raw patch shape before transform: (1, 640, 640)
Raw patch shape after transform: torch.Size([1, 1, 640, 640])


In [22]:
# The issue is that the old dataset was created before fixes
# Let's create a completely fresh dataset by clearing and reimporting everything
import sys
# Remove all pytorch3dunet modules from cache
modules_to_remove = [k for k in sys.modules.keys() if 'pytorch3dunet' in k or 'model_ranking' in k]
for mod in modules_to_remove:
    del sys.modules[mod]

# Reimport
from model_ranking import loader_classes

# Create fresh dataset
dataset_class = loader_classes(loader_config['dataset'])
pred_datasets_new = dataset_class.create_datasets(loader_config, 'test')
if len(pred_datasets_new) > 0:
    dataset_new = pred_datasets_new[0]
    print(f"New dataset created successfully!")
    print(f"New dataset auto_padding: {dataset_new.auto_padding}")
    print(f"New dataset cached raw shape: {dataset_new._raw.shape if dataset_new._raw is not None else 'Not loaded'}")
    
    # Test getting a patch
    raw_new, _ = dataset_new[0]
    print(f"New dataset patch shape: {raw_new.shape}")
else:
    print("Failed to create dataset")

2025-10-29 15:36:42,846 [MainThread] INFO HDF5Dataset - Loading test set from: /scratch/talks/data/FlyWing/GT/test/per03.h5...
2025-10-29 15:36:42,846 [MainThread] INFO HDF5Dataset - Loading test set from: /scratch/talks/data/FlyWing/GT/test/per03.h5...
2025-10-29 15:36:42,846 [MainThread] INFO HDF5Dataset - Loading test set from: /scratch/talks/data/FlyWing/GT/test/per03.h5...
2025-10-29 15:36:42,850 [MainThread] INFO HDF5Dataset - Auto-calculated padding: [0, 1, 0] for patch_shape: [1, 640, 640] and volume_shape: (160, 639, 765)
2025-10-29 15:36:42,850 [MainThread] INFO HDF5Dataset - Auto-calculated padding: [0, 1, 0] for patch_shape: [1, 640, 640] and volume_shape: (160, 639, 765)
2025-10-29 15:36:42,850 [MainThread] INFO HDF5Dataset - Auto-calculated padding: [0, 1, 0] for patch_shape: [1, 640, 640] and volume_shape: (160, 639, 765)
2025-10-29 15:36:42,852 [MainThread] INFO HDF5Dataset - Calculating mean and std of the raw data...
2025-10-29 15:36:42,852 [MainThread] INFO HDF5Datas

In [23]:
# Check the cached raw shape AFTER getting a patch
print(f"Cached raw shape AFTER getting patch: {dataset_new._raw.shape if dataset_new._raw is not None else 'Still not loaded'}")

# Check what the raw_slices look like
print(f"First raw slice: {dataset_new.raw_slices[0]}")

# Manually extract
if dataset_new._raw is not None:
    raw_slice = dataset_new.raw_slices[0]
    extracted = dataset_new._raw[raw_slice]
    print(f"Manually extracted shape: {extracted.shape}")

Cached raw shape AFTER getting patch: Still not loaded
First raw slice: (slice(0, 1, None), slice(0, 640, None), slice(62, 702, None))


In [24]:
# Now test with the fixed code - reload everything again
import sys
modules_to_remove = [k for k in sys.modules.keys() if 'pytorch3dunet' in k or 'model_ranking' in k]
for mod in modules_to_remove:
    del sys.modules[mod]

from model_ranking import loader_classes

# Create fresh dataset with the fix
dataset_class = loader_classes(loader_config['dataset'])
pred_datasets_fixed = dataset_class.create_datasets(loader_config, 'test')
if len(pred_datasets_fixed) > 0:
    dataset_fixed = pred_datasets_fixed[0]
    print(f"Fixed dataset created successfully!")
    print(f"Fixed dataset auto_padding: {dataset_fixed.auto_padding}")
    
    # Test getting a patch
    raw_fixed, _ = dataset_fixed[0]
    print(f"Fixed dataset patch shape: {raw_fixed.shape}")
    print(f"SUCCESS! ✓" if raw_fixed.shape == torch.Size([1, 1, 640, 640]) else f"STILL WRONG ✗")
else:
    print("Failed to create dataset")

2025-10-29 15:39:02,889 [MainThread] INFO HDF5Dataset - Loading test set from: /scratch/talks/data/FlyWing/GT/test/per03.h5...
2025-10-29 15:39:02,889 [MainThread] INFO HDF5Dataset - Loading test set from: /scratch/talks/data/FlyWing/GT/test/per03.h5...
2025-10-29 15:39:02,889 [MainThread] INFO HDF5Dataset - Loading test set from: /scratch/talks/data/FlyWing/GT/test/per03.h5...
2025-10-29 15:39:02,889 [MainThread] INFO HDF5Dataset - Loading test set from: /scratch/talks/data/FlyWing/GT/test/per03.h5...
2025-10-29 15:39:02,894 [MainThread] INFO HDF5Dataset - Auto-calculated padding: [0, 1, 0] for patch_shape: [1, 640, 640] and volume_shape: (160, 639, 765)
2025-10-29 15:39:02,894 [MainThread] INFO HDF5Dataset - Auto-calculated padding: [0, 1, 0] for patch_shape: [1, 640, 640] and volume_shape: (160, 639, 765)
2025-10-29 15:39:02,894 [MainThread] INFO HDF5Dataset - Auto-calculated padding: [0, 1, 0] for patch_shape: [1, 640, 640] and volume_shape: (160, 639, 765)
2025-10-29 15:39:02,894 

NameError: name 'torch' is not defined

In [25]:
# Verify the fix worked
print(f"✓ SUCCESS! The patch shape is now correct: {raw_fixed.shape}")
print(f"Expected: torch.Size([1, 1, 640, 640])")
print(f"Got:      {raw_fixed.shape}")
print(f"Match: {raw_fixed.shape[2] == 640 and raw_fixed.shape[3] == 640}")

# Also verify cached data
print(f"\nCached _raw shape (with auto_padding): {dataset_fixed._raw.shape if dataset_fixed._raw is not None else 'Not loaded'}")
print(f"Cached _raw_padded shape (with auto + halo): {dataset_fixed._raw_padded.shape if dataset_fixed._raw_padded is not None else 'Not loaded'}")

✓ SUCCESS! The patch shape is now correct: torch.Size([1, 1, 640, 640])
Expected: torch.Size([1, 1, 640, 640])
Got:      torch.Size([1, 1, 640, 640])
Match: True

Cached _raw shape (with auto_padding): (160, 641, 765)
Cached _raw_padded shape (with auto + halo): (160, 641, 765)


torch.Size([1, 1, 639, 640])
